# 7 · Train Linear Probing
Builds and runs the shell commands that train a linear head on top of the frozen DINOv2 backbone for each annotation task. Tasks with an existing `results-linear.csv` are skipped automatically.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
DEVICE    = 0
CKP_PATH  = './metadata/DINOv2_checkpoint/'
DATA_PATH = 'PATH_TO_H5_PARTICLES_DIR'                        # <-- set this
CONDA_SH  = '.../miniconda3/etc/profile.d/conda.sh'       # <-- set this
CONDA_ENV = 'cryoPANDA'

EVAL_TASKS = (
    'Symmetry_ue_train',
    'DefocusV_ue_train',
    'MaxDiameter_ue_train',
    'MolecularWeight_kDa_ue_train',
    'ImagePixelSize_ue_train',
    'EMD_Resolution_ue_train',

    'Symmetry_ue_train_on_eval',
    'DefocusV_ue_train_on_eval',
    'MaxDiameter_ue_train_on_eval',
    'MolecularWeight_kDa_ue_train_on_eval',
    'ImagePixelSize_ue_train_on_eval',
    'EMD_Resolution_ue_train_on_eval',
)

In [2]:
import os
import subprocess

## 1 · Build training commands

In [3]:
commands = []
for task in EVAL_TASKS:
    result_csv = os.path.join(CKP_PATH, f'Linear_{task}', 'results-linear.csv')
    if os.path.exists(result_csv):
        print(f'  [skip]   {task}')
        continue
    cmd = (
        f'CUDA_VISIBLE_DEVICES={DEVICE} PYTHONPATH=${{PWD}} '
        f'python dinov3/eval/linear.py '
        f'model.config_file={CKP_PATH}config.yaml '
        f'model.pretrained_weights={CKP_PATH}teacher_checkpoint.pth '
        f'output_dir={CKP_PATH}Linear_{task}/ '
        f'train.dataset=MultiH5:root={DATA_PATH}:h5_key=images'
            f':parquet_annotations=./metadata/parquet_files/train/{task}.parquet '
        f'train.val_dataset=MultiH5:root={DATA_PATH}:h5_key=images'
            f':parquet_annotations=./metadata/parquet_files/eval/{task}.parquet'
    )
    commands.append(cmd)
    print(f'  [queued] {task}')

print(f'\n{len(commands)} task(s) queued')

  [queued] Symmetry_ue_train
  [queued] DefocusV_ue_train
  [queued] MaxDiameter_ue_train
  [queued] MolecularWeight_kDa_ue_train
  [queued] ImagePixelSize_ue_train
  [queued] EMD_Resolution_ue_train
  [queued] Symmetry_ue_train_on_eval
  [queued] DefocusV_ue_train_on_eval
  [queued] MaxDiameter_ue_train_on_eval
  [queued] MolecularWeight_kDa_ue_train_on_eval
  [queued] ImagePixelSize_ue_train_on_eval
  [queued] EMD_Resolution_ue_train_on_eval

12 task(s) queued


## 2 · Run all queued tasks

In [ ]:
if commands:
    # Chain with ' && ' so execution stops on the first failure
    shell_script = (
        f'source {CONDA_SH}\n'
        f'conda activate {CONDA_ENV}\n'
        f'export MKL_THREADING_LAYER=GNU\n'
        + ' && '.join(commands) + '\n'
    )
    result = subprocess.run(['/bin/bash', '-lc', shell_script])
    if result.returncode != 0:
        print(f'WARNING: one or more tasks exited with return code {result.returncode}')
    else:
        print('All tasks completed successfully')
else:
    print('Nothing to run — all tasks already have results')